# Cosine training — 5 model (dataset dedupe_gui) + perbandingan baseline

Notebook ini melatih 5 model dengan varian **cosine** (mengikuti config varian B·cosine pada notebook asli masing-masing model) pada **dataset yang telah diatur ulang menggunakan `Reproduce Code/dedupe_gui.py`** (keputusan keep/remove dari `outputs/results/duplicate_groups_with_decision.csv` / `keep_decisions.json`).

**Model (5) dan config training (sesuai notebook asli, varian cosine):**
- **RepViT-M0.9** (`timm` `repvit_m0_9.dist_450e_in1k`) — LR **2e-3** (ref: `Algorithm Exploration/RepViTM09_ImageNetNorm_Waste_Classification.ipynb`)
- **StarNet-S2** (`timm` `starnet_s2.in1k`) — LR **2e-3** (ref: `Algorithm Exploration/StarNetS2_ImageNetNorm_Waste_Classification.ipynb`)
- **MobileNetV3-Large TV** (`torchvision` `mobilenet_v3_large`) — LR **5e-4** (ref: `Algorithm Exploration/MobileNetV3Large_Torchvision_Waste_Classification.ipynb`)
- **MobileNetV2** (`torchvision` `mobilenet_v2`) — LR **1e-3** (ref: `Reproduce Code/MobileNetV2_Cosine_and_CosineAug_Waste_Classification.ipynb`)
- **MobileNetV3-Small** (`torchvision` `mobilenet_v3_small`) — LR **1e-3** (ref: `Algorithm Exploration/MobileNetV3Small_Waste_Classification.ipynb`)

Resep cosine (sama seperti asli): SGD (momentum 0.9), **CosineAnnealingLR(T_max=50)**, label smoothing 0.1, weight decay 1e-4, augmentasi standar (`RandomResizedCrop`+`RandomHorizontalFlip`), input 224×224 + ImageNet norm, batch 42, 50 epoch, early stopping (patience 10, min 10).

**Di akhir:** perbandingan kelima model tersebut **ditambah baseline CNN dan baseline MobileNetV2** (dimuat dari `outputs/results/cnn_baseline_metrics.json` dan `outputs/results/mobilenetv2_metrics.json` bila tersedia).

> **Catatan split/dataset:** notebook ini membangun kurasi & split 8:1:1 (seed 42, stratified per kelas & source_group) dari keputusan `dedupe_gui` dan menyimpannya ke `outputs/splits/dedupe_gui_{all,train,val,test}.csv` (tidak menimpa `outputs/splits/all.csv`). Baseline CNN/MobileNetV2 yang dimuat dari file metrik existing umumnya berasal dari split/dataset asli (`outputs/splits/all.csv`), sehingga angkanya **tidak langsung apples-to-apples** dengan run dedupe_gui pada notebook ini kecuali dinyatakan lain. Artefak run ini memakai prefix `*_cosine_dedupe_gui` agar tidak menimpa hasil sebelumnya.

## 1. Setup dan dependensi

In [ ]:
# ---------------------------------------------------------------------------
# Auto-install dependencies. Packages that are already installed are untouched.
# ---------------------------------------------------------------------------
from __future__ import annotations

import importlib.util
import subprocess
import sys


def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def pip_install(*packages: str, index_url: str | None = None) -> bool:
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if index_url:
        command += ["--index-url", index_url]
    command += list(packages)
    print("    $", " ".join(command))
    try:
        return subprocess.run(command, check=False).returncode == 0
    except Exception as exc:
        print(f"[warn] pip install failed for {' '.join(packages)}: {exc}")
        return False


def ensure(packages: dict[str, str]) -> list[str]:
    missing = [pkg for pkg, mod in packages.items() if not module_available(mod)]
    if not missing:
        return []
    print(f"[setup] installing: {', '.join(missing)}")
    for package in missing:
        pip_install(package)
    still_missing = [pkg for pkg in missing if not module_available(packages[pkg])]
    if still_missing:
        print("[warn] could not install: " + ", ".join(still_missing))
    return still_missing


ensure({
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scikit-learn": "sklearn",
    "Pillow": "PIL",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
    "timm": "timm",
})

if not module_available("torch") or not module_available("torchvision"):
    print("[setup] installing torch and torchvision")
    installed = pip_install(
        "torch", "torchvision", index_url="https://download.pytorch.org/whl/cpu"
    )
    if not installed and (not module_available("torch") or not module_available("torchvision")):
        print("[setup] retrying with the default PyPI index")
        pip_install("torch", "torchvision")

print("[setup] dependency check done")

In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
try:
    from ipywidgets import IProgress  # noqa: F401
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.unicode_minus": False,
})

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "outputs").is_dir() and (candidate / "Datasets").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "outputs"
SPLIT_DIR = OUTPUT_DIR / "splits"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"
for _d in (OUTPUT_DIR, SPLIT_DIR, MODEL_DIR, RESULT_DIR, FIGURE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(
    os.environ.get("TRASH_DATA_DIR", REPO_ROOT / "Datasets" / "images" / "images")
).resolve()

DECISIONS_CSV = RESULT_DIR / "duplicate_groups_with_decision.csv"
GROUPS_MD5_CSV = RESULT_DIR / "duplicate_groups_md5.csv"

# Protocol defaults (mengikuti notebook asli; LR per-model di-override di MODEL_SPECS)
IMG_SIZE = 224
BATCH_SIZE = 42
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
MOMENTUM = 0.9
STEP_SIZE = 7
GAMMA = 0.1
NUM_WORKERS = 0
PRETRAINED = os.environ.get("TRASH_REPRO_PRETRAINED", "1") == "1"
TEST_BATCH_SIZE = 128
EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_EPOCHS = 10

QUICK_TEST = os.environ.get("TRASH_REPRO_QUICK", "0") == "1"
TRAIN_PER_CLASS = VAL_PER_CLASS = TEST_PER_CLASS = None
if QUICK_TEST:
    NUM_EPOCHS = 2
    BATCH_SIZE = 16
    PRETRAINED = False
    TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 20, 5, 5
    EARLY_STOP_PATIENCE = 2
    EARLY_STOP_MIN_EPOCHS = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python           : {sys.version.split()[0]}")
print(f"PyTorch          : {torch.__version__}")
print(f"Device           : {DEVICE}")
print(f"Repo root        : {REPO_ROOT}")
print(f"Data root        : {DATA_ROOT} (exists={DATA_ROOT.exists()})")
print(f"Quick test       : {QUICK_TEST}")
print(f"Pretrained       : {PRETRAINED}")

## 2. Dataset kurasi dedupe_gui dan split 8:1:1

Membangun set gambar final dari keputusan `dedupe_gui.py`:
- semua file gambar di disk (`Datasets/images/images/<class>/<source>/*`) 
- dikurangi anggota grup duplikat (`duplicate_groups_md5.csv`) 
- ditambah gambar dengan keputusan **keep** (`duplicate_groups_with_decision.csv`).

Lalu split stratified 8:1:1 (seed 42) per `(class_name, source_group)`. Manifest disimpan ke `outputs/splits/dedupe_gui_*.csv`.

In [ ]:
CLASS_NAMES = [
    "aerosol_cans", "aluminum_food_cans", "aluminum_soda_cans",
    "cardboard_boxes", "cardboard_packaging", "clothing",
    "coffee_grounds", "disposable_plastic_cutlery", "eggshells",
    "food_waste", "glass_beverage_bottles", "glass_cosmetic_containers",
    "glass_food_jars", "magazines", "newspaper", "office_paper",
    "paper_cups", "plastic_cup_lids", "plastic_detergent_bottles",
    "plastic_food_containers", "plastic_shopping_bags", "plastic_soda_bottles",
    "plastic_straws", "plastic_trash_bags", "plastic_water_bottles",
    "shoes", "steel_food_cans", "styrofoam_cups",
    "styrofoam_food_containers", "tea_bags",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

MAJOR_NAMES = ["Kitchen waste", "Recyclable waste", "Other waste"]
MAJOR_TO_IDX = {name: i for i, name in enumerate(MAJOR_NAMES)}

CLASS_TO_MAJOR = {
    "coffee_grounds": "Kitchen waste", "eggshells": "Kitchen waste",
    "food_waste": "Kitchen waste", "tea_bags": "Kitchen waste",
    "aerosol_cans": "Recyclable waste", "aluminum_food_cans": "Recyclable waste",
    "aluminum_soda_cans": "Recyclable waste", "cardboard_boxes": "Recyclable waste",
    "cardboard_packaging": "Recyclable waste", "clothing": "Recyclable waste",
    "glass_beverage_bottles": "Recyclable waste", "glass_cosmetic_containers": "Recyclable waste",
    "glass_food_jars": "Recyclable waste", "magazines": "Recyclable waste",
    "newspaper": "Recyclable waste", "office_paper": "Recyclable waste",
    "paper_cups": "Recyclable waste", "plastic_detergent_bottles": "Recyclable waste",
    "plastic_food_containers": "Recyclable waste", "plastic_shopping_bags": "Recyclable waste",
    "plastic_soda_bottles": "Recyclable waste", "plastic_water_bottles": "Recyclable waste",
    "shoes": "Recyclable waste", "steel_food_cans": "Recyclable waste",
    "disposable_plastic_cutlery": "Other waste", "plastic_cup_lids": "Other waste",
    "plastic_straws": "Other waste", "plastic_trash_bags": "Other waste",
    "styrofoam_cups": "Other waste", "styrofoam_food_containers": "Other waste",
}
assert not (set(CLASS_NAMES) - set(CLASS_TO_MAJOR)), "Missing major-class mapping"
print(f"Classes: {NUM_CLASSES}")

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def scan_all_images(root: Path) -> set[str]:
    paths = set()
    for dirpath, _, files in os.walk(root):
        for f in files:
            if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
                p = Path(dirpath) / f
                try:
                    paths.add(p.relative_to(root).as_posix())
                except ValueError:
                    pass
    return paths


def infer_source_group(rel_path: str) -> str:
    parts = Path(rel_path).parts
    for p in parts:
        if p in {"default", "real_world"}:
            return p
    return "unknown"


def stratified_split(frame: pd.DataFrame, train_ratio=0.8, val_ratio=0.1, seed=SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    split_series = pd.Series(index=frame.index, dtype="object")
    for (_, _), group in frame.groupby(["class_name", "source_group"], sort=True):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)
        n = len(indices)
        if n >= 3:
            n_train = int(round(n * train_ratio))
            n_val = int(round(n * val_ratio))
            n_train = max(1, min(n_train, n - 2))
            n_val = max(1, min(n_val, n - n_train - 1))
        elif n == 2:
            n_train, n_val = 1, 0
        else:
            n_train, n_val = 1, 0
        split_series.loc[indices[:n_train]] = "train"
        if n_val:
            split_series.loc[indices[n_train:n_train + n_val]] = "val"
        split_series.loc[indices[n_train + n_val:]] = "test"
    result = frame.copy()
    result["split"] = split_series
    if result["split"].isna().any():
        raise RuntimeError("Some images were not assigned to a split.")
    return result


assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}. Set TRASH_DATA_DIR."
missing_dirs = [name for name in CLASS_NAMES if not (DATA_ROOT / name).is_dir()]
assert not missing_dirs, f"Missing class folders under {DATA_ROOT}: {missing_dirs}"

all_disk = scan_all_images(DATA_ROOT)
print(f"Images on disk: {len(all_disk):,}")

group_members = set()
if GROUPS_MD5_CSV.exists():
    with GROUPS_MD5_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            group_members.add(row["rel_path"])
elif DECISIONS_CSV.exists():
    with DECISIONS_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            group_members.add(row["rel_path"])
print(f"In duplicate groups: {len(group_members):,}")

kept, removed = set(), set()
if DECISIONS_CSV.exists():
    with DECISIONS_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            (kept if row.get("decision") == "keep" else removed).add(row["rel_path"])
    print(f"dedupe_gui decisions - keep: {len(kept):,}, remove: {len(removed):,}")
else:
    print(f"[warn] {DECISIONS_CSV} not found; using all disk images.")

if group_members:
    curated_rel = (all_disk - group_members) | kept
else:
    curated_rel = set(all_disk)
curated_rel = {p for p in curated_rel if (DATA_ROOT / p).exists() and Path(p).parts[0] in CLASS_NAMES}
print(f"Curated images (dedupe_gui): {len(curated_rel):,}")

records = []
for rel in sorted(curated_rel):
    class_name = Path(rel).parts[0]
    source_group = infer_source_group(rel)
    records.append({
        "abs_path": str((DATA_ROOT / rel).resolve()),
        "rel_path": rel,
        "class_name": class_name,
        "source_group": source_group,
    })

df = pd.DataFrame(records)
assert not df.empty, "No curated images found."
df["label"] = df["class_name"].map(CLASS_TO_IDX).astype(int)

df = stratified_split(df, train_ratio=0.8, val_ratio=0.1, seed=SEED)

if QUICK_TEST:
    limits = {"train": TRAIN_PER_CLASS, "val": VAL_PER_CLASS, "test": TEST_PER_CLASS}
    parts = []
    for (split_name, _class_name), group in df.groupby(["split", "class_name"], sort=True):
        n_total = limits[split_name]
        n_groups = max(1, group["source_group"].nunique())
        n_per_group = max(1, math.ceil(n_total / n_groups))
        for _source_group, subgroup in group.groupby("source_group", sort=True):
            parts.append(subgroup.head(n_per_group))
    df = pd.concat(parts, ignore_index=True)
    trimmed = []
    for (split_name, _class_name), group in df.groupby(["split", "class_name"], sort=True):
        trimmed.append(group.head(limits[split_name]))
    df = pd.concat(trimmed, ignore_index=True)

df = df.sort_values(["split", "class_name", "rel_path"]).reset_index(drop=True)

SPLIT_ALL = SPLIT_DIR / "dedupe_gui_all.csv"
df.to_csv(SPLIT_ALL, index=False)
for split_name in ("train", "val", "test"):
    df[df["split"] == split_name].reset_index(drop=True).to_csv(
        SPLIT_DIR / f"dedupe_gui_{split_name}.csv", index=False
    )

print("\nCurated class counts:")
display(df["class_name"].value_counts().reindex(CLASS_NAMES).rename("images").to_frame().T)
print("\nSplit sizes:")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print("\nSource groups:")
print(df["source_group"].value_counts().to_string())
print(f"\nSplit manifests written: {SPLIT_ALL}")

assert df.groupby("split")["class_name"].nunique().eq(NUM_CLASSES).all(), "Every split must contain every class."
assert not df["rel_path"].duplicated().any(), "Duplicate rel_path in curated manifest."

## 3. Transform, dataset, dan loader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TrashDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, root: Path, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        with Image.open(self.root / row["rel_path"]) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(frame: pd.DataFrame, training: bool, batch_size: int, transform=None) -> DataLoader:
    dataset = TrashDataset(
        frame, DATA_ROOT,
        transform=transform if transform is not None else (train_transform if training else eval_transform),
    )
    drop_last = training and len(dataset) >= batch_size and len(dataset) % batch_size == 1
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=training,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        drop_last=drop_last, persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker, generator=generator,
    )


train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

train_loader = make_loader(train_df, training=True, batch_size=BATCH_SIZE)
val_loader = make_loader(val_df, training=False, batch_size=BATCH_SIZE)
test_loader = make_loader(test_df, training=False, batch_size=TEST_BATCH_SIZE)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 4. Definisi 5 model (config mengikuti notebook asli, varian cosine)

Config training per model mengikuti varian **B · cosine** pada notebook asli:
- **RepViT-M0.9** / **StarNet-S2**: SGD lr **2e-3**, momentum 0.9, wd 1e-4, CosineAnnealingLR, label smoothing 0.1, augmen standar
- **MobileNetV3-Large TV**: SGD lr **5e-4**, momentum 0.9, wd 1e-4, CosineAnnealingLR, label smoothing 0.1, augmen standar
- **MobileNetV2** / **MobileNetV3-Small**: SGD lr **1e-3**, momentum 0.9, wd 1e-4, CosineAnnealingLR, label smoothing 0.1, augmen standar

(50 epoch, batch 42, early stopping patience 10 min 10, pretrained.)

In [ ]:
import timm


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_mobilenet_v2(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


def build_mobilenet_v3_small(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    return model


def build_mobilenet_v3_large_tv(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v3_large(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    return model


def build_repvit_m0_9(num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(
        "repvit_m0_9.dist_450e_in1k", pretrained=pretrained, num_classes=num_classes
    )


def build_starnet_s2(num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(
        "starnet_s2.in1k", pretrained=pretrained, num_classes=num_classes
    )


# Config per model mengikuti varian B·cosine di notebook asli masing-masing.
MODEL_SPECS = {
    "repvit_m0_9_in1k_cosine_dedupe_gui": {
        "label": "RepViT-M0.9 (cosine, dedupe_gui)",
        "arch": "repvit_m0_9",
        "builder": build_repvit_m0_9,
        "params_m": 4.742428, "gmacs": 0.816535872,
        # asli: Algorithm Exploration/RepViTM09_ImageNetNorm_Waste_Classification.ipynb
        "learning_rate": 2e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "starnet_s2_in1k_cosine_dedupe_gui": {
        "label": "StarNet-S2 (cosine, dedupe_gui)",
        "arch": "starnet_s2",
        "builder": build_starnet_s2,
        "params_m": 3.432734, "gmacs": 0.543865344,
        # asli: Algorithm Exploration/StarNetS2_ImageNetNorm_Waste_Classification.ipynb
        "learning_rate": 2e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "mobilenetv3large_tv_cosine_dedupe_gui": {
        "label": "MobileNetV3-Large TV (cosine, dedupe_gui)",
        "arch": "mobilenet_v3_large_tv",
        "builder": build_mobilenet_v3_large_tv,
        "params_m": 4.240462, "gmacs": 0.21534816,
        # asli: Algorithm Exploration/MobileNetV3Large_Torchvision_Waste_Classification.ipynb
        "learning_rate": 5e-4, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "mobilenetv2_cosine_dedupe_gui": {
        "label": "MobileNetV2 (cosine, dedupe_gui)",
        "arch": "mobilenet_v2",
        "builder": build_mobilenet_v2,
        "params_m": 2.262302, "gmacs": 0.299532672,
        # asli: Reproduce Code/MobileNetV2_Cosine_and_CosineAug_Waste_Classification.ipynb
        "learning_rate": 1e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "mobilenetv3small_cosine_dedupe_gui": {
        "label": "MobileNetV3-Small (cosine, dedupe_gui)",
        "arch": "mobilenet_v3_small",
        "builder": build_mobilenet_v3_small,
        "params_m": 1.548606, "gmacs": 0.05551712,
        # asli: Algorithm Exploration/MobileNetV3Small_Waste_Classification.ipynb
        "learning_rate": 1e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
}

for key, spec in MODEL_SPECS.items():
    print(
        f"{key:45s} | lr={spec['learning_rate']:<7g} | sched={spec['scheduler']:<6} | "
        f"ls={spec['label_smoothing']} | wd={spec['weight_decay']} | aug={spec['augmentation']}"
    )

## 5. Training dan evaluasi (utilitas)

In [ ]:
def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)
    except Exception:
        return torch.load(path, map_location=device, weights_only=False)


def train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs, device, checkpoint_path, history_path, model_name, arch,
    extra_config=None, full_model_path=None,
    early_stopping_patience=None, early_stop_min_epochs=0,
):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    dataloaders = {"train": train_loader, "val": val_loader}
    dataset_sizes = {phase: len(loader.dataset) for phase, loader in dataloaders.items()}

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        epoch_start = time.time()
        epoch_losses, epoch_accs = {}, {}
        improved_this_epoch = False

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0
            loader = dataloaders[phase]
            progress = tqdm(total=dataset_sizes[phase], desc=f"{model_name} {phase} {epoch}/{epochs}", leave=False)
            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                progress.update(inputs.size(0))
            progress.close()

            if phase == "train" and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = (running_corrects.double() / dataset_sizes[phase]).item()
            epoch_losses[phase] = float(epoch_loss)
            epoch_accs[phase] = float(epoch_acc)
            print(f"  {phase} loss {epoch_loss:.4f} acc {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_val_acc:
                best_val_acc = float(epoch_acc)
                best_epoch = epoch
                improved_this_epoch = True
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save({
                    "arch": arch, "model_name": model_name, "state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES, "major_names": MAJOR_NAMES,
                    "class_to_major": CLASS_TO_MAJOR, "img_size": IMG_SIZE,
                    "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
                    "epoch": epoch, "val_acc": float(epoch_acc), "val_loss": float(epoch_loss),
                    "config": {"seed": SEED, "batch_size": BATCH_SIZE, "epochs": epochs,
                               "pretrained": PRETRAINED, **(extra_config or {})},
                }, checkpoint_path)

        epochs_without_improvement = 0 if improved_this_epoch else epochs_without_improvement + 1
        current_lr = optimizer.param_groups[0]["lr"]
        record = {
            "epoch": epoch, "train_loss": epoch_losses["train"], "train_acc": epoch_accs["train"],
            "val_loss": epoch_losses["val"], "val_acc": epoch_accs["val"],
            "lr": float(current_lr), "time_sec": float(time.time() - epoch_start), "best_val_acc": best_val_acc,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(history_path, index=False)

        if (early_stopping_patience is not None and epoch >= early_stop_min_epochs
                and epochs_without_improvement >= early_stopping_patience):
            print(f"Early stopping at epoch {epoch} (no improvement for {early_stopping_patience} epochs).")
            break

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s | best val {best_val_acc:.4f} @ {best_epoch}")
    model.load_state_dict(best_model_wts)
    if full_model_path is not None:
        torch.save(model, full_model_path)
    return history, best_val_acc, best_epoch


def aggregate_major_probs(probs: np.ndarray, class_names):
    major_probs = np.zeros((probs.shape[0], len(MAJOR_NAMES)), dtype=np.float64)
    for class_idx, class_name in enumerate(class_names):
        major_probs[:, MAJOR_TO_IDX[CLASS_TO_MAJOR[class_name]]] += probs[:, class_idx]
    return major_probs


@torch.no_grad()
def predict_loader(model, loader, criterion, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0
    total_samples = 0
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return probs, labels, total_loss / max(total_samples, 1)


def evaluate_and_save(model, loader, criterion, device, frame, model_key):
    probs, labels, loss = predict_loader(model, loader, criterion, device)
    preds = probs.argmax(axis=1)
    report_dict = classification_report(
        labels, preds, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    cm = confusion_matrix(labels, preds, labels=np.arange(NUM_CLASSES))
    per_class_acc = np.divide(cm.diagonal(), cm.sum(axis=1), out=np.zeros(NUM_CLASSES, dtype=np.float64),
                              where=cm.sum(axis=1) != 0)

    major_probs = aggregate_major_probs(probs, CLASS_NAMES)
    major_true = frame["class_name"].map(lambda c: MAJOR_TO_IDX[CLASS_TO_MAJOR[c]]).to_numpy()
    major_pred = np.array([MAJOR_TO_IDX[CLASS_TO_MAJOR[CLASS_NAMES[int(i)]]] for i in preds])
    major_prob_pred = major_probs.argmax(axis=1)
    major_report = classification_report(
        major_true, major_pred, labels=np.arange(len(MAJOR_NAMES)),
        target_names=MAJOR_NAMES, output_dict=True, zero_division=0,
    )
    major_cm = confusion_matrix(major_true, major_pred, labels=np.arange(len(MAJOR_NAMES)))

    pred_frame = frame[["rel_path", "class_name", "source_group"]].copy()
    pred_frame["true_label"] = labels
    pred_frame["pred_label"] = preds
    pred_frame["pred_class"] = [CLASS_NAMES[i] for i in preds]
    pred_frame["confidence"] = probs.max(axis=1)
    pred_frame["correct"] = (labels == preds).astype(int)
    pred_frame.to_csv(RESULT_DIR / f"{model_key}_predictions.csv", index=False)

    metrics = {
        "model": model_key, "split": "test", "num_test": int(len(labels)),
        "accuracy": float(accuracy_score(labels, preds)), "loss": float(loss),
        "top3_accuracy": float(top_k_accuracy_score(labels, probs, k=min(3, NUM_CLASSES), labels=np.arange(NUM_CLASSES))),
        "macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "weighted_f1": float(report_dict["weighted avg"]["f1-score"]),
        "per_class_accuracy": {name: float(per_class_acc[i]) for i, name in enumerate(CLASS_NAMES)},
        "group_accuracy": {str(g): float(sub["correct"].mean()) for g, sub in pred_frame.groupby("source_group")},
        "classification_report": report_dict, "confusion_matrix": cm.tolist(),
        "major_accuracy": float(accuracy_score(major_true, major_pred)),
        "major_prob_accuracy": float(accuracy_score(major_true, major_prob_pred)),
        "major_report": major_report, "major_confusion_matrix": major_cm.tolist(),
        "major_class_names": MAJOR_NAMES, "class_names": CLASS_NAMES,
    }
    with open(RESULT_DIR / f"{model_key}_metrics.json", "w") as f:
        json.dump(jsonable(metrics), f, indent=2)
    pd.DataFrame(report_dict).T.to_csv(RESULT_DIR / f"{model_key}_classification_report.csv")
    pd.DataFrame(major_report).T.to_csv(RESULT_DIR / f"{model_key}_major_report.csv")
    pd.DataFrame(per_class_acc, index=CLASS_NAMES, columns=["accuracy"]).to_csv(
        RESULT_DIR / f"{model_key}_per_class_accuracy.csv"
    )
    mis = pred_frame[~pred_frame["correct"].astype(bool)].copy()
    mis.to_csv(RESULT_DIR / f"{model_key}_misclassified.csv", index=False)

    print(f"{model_key}: acc {metrics['accuracy']*100:.2f}% | loss {metrics['loss']:.4f} | "
          f"macroF1 {metrics['macro_f1']:.4f} | top3 {metrics['top3_accuracy']*100:.2f}% | "
          f"3-class {metrics['major_accuracy']*100:.2f}%")
    return metrics, pred_frame


def plot_confusion_matrix(cm, labels, title, output_path, normalize=True):
    cm = np.asarray(cm, dtype=np.float64)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        plot_cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        fmt = ".2f"
    else:
        plot_cm = cm
        fmt = "d"
    annot = len(labels) <= 12
    fig, ax = plt.subplots(figsize=(max(7, len(labels) * 0.48), max(6, len(labels) * 0.42)))
    sns.heatmap(plot_cm, annot=annot, fmt=fmt, cmap="Blues", xticklabels=labels, yticklabels=labels,
                ax=ax, cbar=True, linewidths=0.2 if annot else 0)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    ax.tick_params(axis="x", rotation=90, labelsize=7); ax.tick_params(axis="y", rotation=0, labelsize=7)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_training_curves(history, title: str, output_path: Path):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(frame["epoch"], frame["train_loss"], marker="o", ms=3, label="Train")
    axes[0].plot(frame["epoch"], frame["val_loss"], marker="o", ms=3, label="Validation")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")
    axes[1].plot(frame["epoch"], frame["train_acc"], marker="o", ms=3, label="Train")
    axes[1].plot(frame["epoch"], frame["val_acc"], marker="o", ms=3, label="Validation")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 6. Training 5 model (cosine, dataset dedupe_gui)

In [ ]:
runs = {}


def build_criterion(spec, train_frame):
    """Mengikuti pola notebook asli: class_weight opsional, label smoothing."""
    if spec.get("class_weight"):
        counts = train_frame["class_name"].value_counts().reindex(CLASS_NAMES).to_numpy()
        weights = torch.tensor(
            counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32, device=DEVICE
        )
        return nn.CrossEntropyLoss(weight=weights, label_smoothing=spec.get("label_smoothing", 0.0))
    return nn.CrossEntropyLoss(label_smoothing=spec.get("label_smoothing", 0.0))


def build_scheduler(optimizer, spec, epochs):
    """Mengikuti pola notebook asli."""
    if spec.get("scheduler") == "cosine":
        return lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)


for name, spec in MODEL_SPECS.items():
    print("=" * 80)
    print(f"Training: {spec['label']} ({name})")
    print(f"  config (asli): lr={spec.get('learning_rate')} momentum={spec.get('momentum')} "
          f"scheduler={spec.get('scheduler')} ls={spec.get('label_smoothing')} "
          f"wd={spec.get('weight_decay')} class_weight={spec.get('class_weight')} "
          f"aug={spec.get('augmentation')}")
    print("=" * 80)

    seed_everything(SEED)
    model = spec["builder"](NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)

    lr = spec.get("learning_rate", LEARNING_RATE)
    momentum = spec.get("momentum", MOMENTUM)
    weight_decay = spec.get("weight_decay", 1e-4)
    label_smoothing = spec.get("label_smoothing", 0.1)

    criterion = build_criterion(spec, train_df)
    optimizer = optim.SGD(
        model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay,
    )
    scheduler = build_scheduler(optimizer, spec, NUM_EPOCHS)

    checkpoint_path = MODEL_DIR / f"{name}_trash30_best.pt"
    full_model_path = MODEL_DIR / f"{name}_trash30_full.pt"
    history_path = RESULT_DIR / f"{name}_history.csv"

    history, best_val_acc, best_epoch = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimizer=optimizer, scheduler=scheduler,
        epochs=NUM_EPOCHS, device=DEVICE,
        checkpoint_path=checkpoint_path, full_model_path=full_model_path, history_path=history_path,
        model_name=name, arch=spec["arch"],
        extra_config={
            "dataset": "dedupe_gui_curated",
            "split_manifest": str(SPLIT_ALL),
            "learning_rate": lr,
            "momentum": momentum,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "scheduler": spec.get("scheduler", "cosine"),
            "class_weight": spec.get("class_weight", False),
            "augmentation": spec.get("augmentation", "randomresizedcrop+flip"),
        },
        early_stopping_patience=EARLY_STOP_PATIENCE,
        early_stop_min_epochs=EARLY_STOP_MIN_EPOCHS,
    )

    plot_training_curves(history, f"{spec['label']} — training curves", FIGURE_DIR / f"{name}_curves.png")

    eval_model = spec["builder"](NUM_CLASSES, pretrained=False).to(DEVICE)
    ckpt = load_checkpoint(checkpoint_path, DEVICE)
    eval_model.load_state_dict(ckpt["state_dict"])
    eval_model.eval()
    eval_criterion = nn.CrossEntropyLoss()
    metrics, pred_frame = evaluate_and_save(eval_model, test_loader, eval_criterion, DEVICE, test_df, name)

    plot_confusion_matrix(np.array(metrics["confusion_matrix"]), CLASS_NAMES,
                          f"{spec['label']} — 30-class CM", FIGURE_DIR / f"{name}_confusion_30.png")
    plot_confusion_matrix(np.array(metrics["major_confusion_matrix"]), MAJOR_NAMES,
                          f"{spec['label']} — 3 major class CM", FIGURE_DIR / f"{name}_confusion_major.png")

    runs[name] = {
        "label": spec["label"], "arch": spec["arch"],
        "config": {
            "learning_rate": lr, "momentum": momentum, "weight_decay": weight_decay,
            "label_smoothing": label_smoothing, "scheduler": spec.get("scheduler", "cosine"),
            "class_weight": spec.get("class_weight", False),
            "augmentation": spec.get("augmentation", "randomresizedcrop+flip"),
        },
        "params_m": spec.get("params_m") or (count_parameters(eval_model) / 1e6),
        "gmacs": spec.get("gmacs"),
        "history": history, "best_val_acc": best_val_acc, "best_epoch": best_epoch,
        "metrics": metrics,
        "checkpoint": str(checkpoint_path),
    }
    print()

print("Selesai training:", list(runs.keys()))

## 7. Perbandingan 5 model (cosine, dedupe_gui) + baseline CNN & MobileNetV2

In [ ]:
def row_from_metrics(label, metrics, params_m=None, gmacs=None, dataset_note="", best_epoch=None, lr=None):
    if metrics is None:
        return None
    major = metrics.get("major_report", {}) or {}
    macro_major = major.get("macro avg", {}).get("f1-score")
    return {
        "Model": label,
        "Dataset/Split": dataset_note,
        "LR": lr,
        "Best epoch": best_epoch,
        "Test acc (%)": (metrics.get("accuracy") or 0) * 100,
        "Loss": metrics.get("loss"),
        "Top-3 (%)": (metrics.get("top3_accuracy") or 0) * 100,
        "Macro F1 (30)": metrics.get("macro_f1"),
        "Macro F1 (3 major)": macro_major,
        "3-class acc (%)": (metrics.get("major_accuracy") or 0) * 100,
        "3-class prob acc (%)": (metrics.get("major_prob_accuracy") or 0) * 100,
        "Params (M)": params_m,
        "GMACs": gmacs,
    }


rows = []
for name, run in runs.items():
    rows.append(row_from_metrics(
        run["label"], run["metrics"],
        params_m=run.get("params_m"), gmacs=run.get("gmacs"),
        dataset_note="dedupe_gui curated (this run)",
        best_epoch=run.get("best_epoch"),
        lr=(run.get("config") or {}).get("learning_rate"),
    ))

# Baselines dari file metrik existing (umumnya split asli outputs/splits/all.csv)
baseline_note = "original split (existing baseline)"

cnn_path = RESULT_DIR / "cnn_baseline_metrics.json"
if cnn_path.exists():
    with open(cnn_path) as f:
        cnn_m = json.load(f)
    cnn_params = None
    try:
        with open(RESULT_DIR / "summary.json") as f:
            s = json.load(f)
        cnn_params = (s.get("results", {}).get("cnn", {}) or {}).get("parameters")
        if cnn_params is not None:
            cnn_params = cnn_params / 1e6
    except Exception:
        pass
    rows.append(row_from_metrics("CNN (baseline)", cnn_m, params_m=cnn_params, gmacs=None,
                                 dataset_note=baseline_note, best_epoch=None, lr=None))
else:
    print(f"[warn] baseline not found: {cnn_path}")

mnv2_path = RESULT_DIR / "mobilenetv2_metrics.json"
if mnv2_path.exists():
    with open(mnv2_path) as f:
        mnv2_m = json.load(f)
    rows.append(row_from_metrics("MobileNetV2 (baseline)", mnv2_m, params_m=2.262302, gmacs=0.299532672,
                                 dataset_note=baseline_note, best_epoch=None, lr=None))
else:
    print(f"[warn] baseline not found: {mnv2_path}")

comp_df = pd.DataFrame([r for r in rows if r is not None])
# sort: cosine dedupe_gui models first by acc desc, then baselines
is_run = comp_df["Dataset/Split"].str.contains("dedupe_gui", na=False)
comp_df = pd.concat([
    comp_df[is_run].sort_values("Test acc (%)", ascending=False),
    comp_df[~is_run],
], ignore_index=True)

display(comp_df.round(4))

comp_csv = RESULT_DIR / "dedupe_gui_cosine_5models_plus_baselines_comparison.csv"
comp_df.to_csv(comp_csv, index=False)
print("Saved:", comp_csv)

# Plot: 5 cosine models only (apples-to-apples on dedupe_gui)
run_df = comp_df[comp_df["Dataset/Split"].str.contains("dedupe_gui", na=False)].copy()
if not run_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    x = np.arange(len(run_df))
    labels = [l.replace(" (cosine, dedupe_gui)", "") for l in run_df["Model"]]
    for ax, col in zip(axes, ["Test acc (%)", "Macro F1 (30)", "3-class acc (%)"]):
        vals = run_df[col].astype(float).values
        ax.bar(x, vals, color="#4C78A8")
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
        ax.set_title(col)
        for i, v in enumerate(vals):
            if pd.notna(v):
                ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    fig.suptitle("5 model — cosine (config asli per model), dataset dedupe_gui (this run)")
    fig.tight_layout()
    fig_path = FIGURE_DIR / "dedupe_gui_cosine_5models_comparison.png"
    fig.savefig(fig_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", fig_path)

## 8. Simpan ringkasan

In [ ]:
summary = {
    "run": "cosine_5models_dedupe_gui",
    "dataset": {
        "curation": "dedupe_gui (duplicate_groups_with_decision.csv / keep_decisions.json)",
        "split_manifests": {
            "all": str(SPLIT_ALL),
            "train": str(SPLIT_DIR / 'dedupe_gui_train.csv'),
            "val": str(SPLIT_DIR / 'dedupe_gui_val.csv'),
            "test": str(SPLIT_DIR / 'dedupe_gui_test.csv'),
        },
        "num_images": int(len(df)),
        "split_sizes": {s: int((df["split"] == s).sum()) for s in ("train", "val", "test")},
    },
    "recipe_note": "Config mengikuti varian B·cosine notebook asli per model (LR berbeda: RepViT/StarNet-S2 2e-3, MNV3-Large TV 5e-4, MNV2/MNV3-Small 1e-3).",
    "training_defaults": {
        "optimizer": "SGD", "momentum": MOMENTUM,
        "scheduler": "cosine (CosineAnnealingLR, T_max=NUM_EPOCHS)",
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "augmentation": "randomresizedcrop+flip",
        "epochs": NUM_EPOCHS,
        "early_stopping_patience": EARLY_STOP_PATIENCE,
        "early_stop_min_epochs": EARLY_STOP_MIN_EPOCHS,
        "batch_size": BATCH_SIZE,
        "test_batch_size": TEST_BATCH_SIZE,
        "img_size": IMG_SIZE,
        "seed": SEED,
    },
    "models": {name: {
        "label": run["label"], "arch": run["arch"],
        "config": run.get("config"),
        "best_epoch": run.get("best_epoch"), "best_val_acc": run.get("best_val_acc"),
        "metrics": {
            "accuracy": run["metrics"]["accuracy"],
            "loss": run["metrics"]["loss"],
            "top3_accuracy": run["metrics"]["top3_accuracy"],
            "macro_f1": run["metrics"]["macro_f1"],
            "major_accuracy": run["metrics"]["major_accuracy"],
        },
        "checkpoint": run.get("checkpoint"),
    } for name, run in runs.items()},
    "comparison_csv": str(comp_csv),
    "note": "Baseline CNN & MobileNetV2 rows (if included) are loaded from existing metrics files and may use the original split; 5 cosine models use dedupe_gui curated split.",
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": __import__('torchvision').__version__,
        "timm": __import__('timm').__version__,
    },
}
summary_path = RESULT_DIR / "dedupe_gui_cosine_5models_plus_baselines_summary.json"
with open(summary_path, "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved:", summary_path)
print("Done.")